# GF

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from IPython.display import clear_output
import openmm
from openmm import unit
import bgflow as bg
import bgmol
from bgflow.utils.types import assert_numpy
kB = unit.MOLAR_GAS_CONSTANT_R.value_in_unit(unit.kilojoule_per_mole/unit.kelvin)
ctx = {'dtype': torch.float32, 'device': torch.device('cuda:1') if torch.cuda.is_available() else torch.device('cpu')}
main = (__name__ == "__main__")

from bgmol.datasets.base import DataSet
from bgmol.api import system_by_name
import mdtraj as md
import os
from bgmol.systems.base import OpenMMSystem
from bgmol.util.importing import import_openmm
_, unit, app = import_openmm()

In [ ]:
class AlaImplicit(OpenMMSystem):
    def __init__(
            self,
    ):
        super(AlaImplicit, self).__init__()

        forcefield = app.ForceField("amber14-all.xml", 'implicit/gbn2.xml')
        pdb = app.PDBFile("ala_no_water.pdb")

        self._system  = forcefield.createSystem(
            pdb.topology,
            nonbondedMethod=app.NoCutoff, 
            constraints=app.HBonds,  
            hydrogenMass=4.0 * unit.amu  
        )

        self._positions = pdb.getPositions(asNumpy=True).value_in_unit(unit.nanometer)
        self._topology = pdb.topology


class Ala2Implicit300(DataSet):

    def __init__(self, root=os.getcwd(), download: bool = False, read: bool = False):
        super(Ala2Implicit300, self).__init__(root=root, download=download, read=read)
        self._system = AlaImplicit()
        self._temperature = 300

    @property
    def trajectory_file(self):
        return os.path.join(self.root, "/home/dell/workstations/hsj_bgflow/experiment/ala/Ala2Tip3p300/trajectory_no_water.dcd")

    @property
    def top_file(self):
        return os.path.join(self.root, "ala_no_water.pdb")

    def read(self, n_frames=None, stride=None, atom_indices=None):
        self.trajectory = md.load(self.trajectory_file, top = self.top_file)


class Ala2Implicit1000(DataSet):

    def __init__(self, root=os.getcwd(), download: bool = False, read: bool = False):
        super(Ala2Implicit1000, self).__init__(root=root, download=download, read=read)
        self._system = AlaImplicit()
        self._temperature = 1000

    @property
    def trajectory_file(self):
        return os.path.join(self.root, "/home/dell/workstations/hsj_bgflow/experiment/ala/Ala2Tip3p1000/trajectory_aligned_no_water.dcd")

    @property
    def top_file(self):
        return os.path.join(self.root, "ala_no_water.pdb")

    def read(self, n_frames=None, stride=None, atom_indices=None):
        self.trajectory = md.load(self.trajectory_file, top = self.top_file)

In [ ]:
is_data_here = True
dataset_1 = Ala2Implicit1000(download=(not is_data_here), read=True)
system_1 = dataset_1.system
coordinates_1 = dataset_1.coordinates
T_1= dataset_1.temperature
dataset_2 = Ala2Implicit300(download=(not is_data_here), read=False)
system_2 = dataset_2.system
coordinates_2 = dataset_2.coordinates
T_2= dataset_2.temperature

In [ ]:
coordinates_1.shape

In [ ]:
model = dataset_1.system

In [ ]:
target_energy_1 = dataset_1.get_energy_model(n_workers=8)

In [ ]:
data_gen = coordinates_1[:500000]
data_len = len(data_gen)

len_training = int(data_len / 20)
subset_data = data_gen[:len_training]

subset_len = len(subset_data)
indices = torch.randperm(subset_len)

len_train = int(subset_len * 0.8)
train_indices = indices[:len_train]
test_indices = indices[len_train:]

training_data_gen = torch.tensor(subset_data[train_indices]).to(**ctx)
test_data = torch.tensor(subset_data[test_indices]).to(**ctx)

print("Training data shape:", training_data_gen.shape)
print("Testing data shape:", test_data.shape)

In [ ]:
#some plotting functions
from matplotlib.colors import LogNorm

def get_phi_psi(trajectory, i=-1, model=model):
    if not isinstance(trajectory, md.Trajectory):
        if isinstance(trajectory, torch.Tensor):
            trajectory = assert_numpy(trajectory.view(len(trajectory), *model.positions.shape))
        trajectory = md.Trajectory(trajectory, model.mdtraj_topology)
    phi = md.compute_phi(trajectory)[1][:,i]
    psi = md.compute_psi(trajectory)[1][:,i]
    return phi, psi

def plot_rama_traj(trajectory, w=None, get_phi=False, i=-1, model=model):
    phi, psi = get_phi_psi(trajectory, i)
    plot_range = [-np.pi, np.pi]
    
    #histogram
    plt.figure(figsize=(14, 4))
    plt.subplot(1,2,1)
    plt.title("Histogram")
    plt.hist2d(phi, psi, 60, weights=w, norm=LogNorm(), range=[plot_range,plot_range])
    plt.xlim(plot_range)
    plt.ylim(plot_range)
    plt.xlabel("$\phi$")
    plt.ylabel("$\psi$")
    plt.gca().set_box_aspect(1)
    
    #trajectory
    plt.subplot(1,2,2)
    plt.title("Trajectory")
    end = len(phi)
    plt.scatter(range(end), phi, c=psi, s=10)
    plt.xlim([0, end])
    plt.xlabel("n_iter")
    plt.ylabel("$\phi$")
    plt.gca().set_box_aspect(0.5)
    plt.show()
    
    if get_phi:
        return phi

def plot_phi_FES(phi, w=None, n_bins='auto', T_high=T_1, T_low=T_2, ymax=20):
    plot_range = (-np.pi, np.pi)
    hist, edges = np.histogram(phi, bins=n_bins, range=plot_range)
    fes_estimate = -np.log(np.where(hist!=0,hist/hist.max(),np.nan))
    plt.plot(edges[:-1]+(edges[1]-edges[0])/2, fes_estimate, label="direct FES")
    DeltaF_T_high = None
    DeltaF_T_low = None
    try:
        ref_file = f'../FESreference/FES-{prior_name}-T{T_1}.dat'
        phi_ref, fes_ref = np.loadtxt(ref_file, usecols=(0,1), unpack=True)
        fes_ref /= (kB*T_high)
        plt.plot(phi_ref, fes_ref, label=f"reference {T_high}K", linestyle='dotted')
        ymax = max(ymax, np.amax(fes_ref))
        DeltaF_T_high = np.logaddexp.reduce(-fes_ref[phi_ref<0])-np.logaddexp.reduce(-fes_ref[phi_ref>0])
    except IOError:
        print('+++ Ref. file not found: '+ref_file+' +++')
    if w is not None:
        if n_bins == 'auto':
            n_bins = 50
        hist, edges =np.histogram(phi, bins=n_bins, range=plot_range, weights=w)
        fes_estimate = -np.log(np.where(hist!=0,hist/hist.max(),np.nan))
        plt.plot(edges[:-1]+(edges[1]-edges[0])/2, fes_estimate, label="reweighted FES")
        try:
            ref_file = f'../FESreference/FES-{target_name}-T{T_low}.dat'
            phi_ref, fes_ref = np.loadtxt(ref_file, usecols=(0,1), unpack=True)
            fes_ref /= (kB*T_low)
            plt.plot(phi_ref, fes_ref, label=f"reference {T_low}K", linestyle='dotted')
            ymax = max(ymax, np.amax(fes_ref))
            DeltaF_T_low = np.logaddexp.reduce(-fes_ref[phi_ref<0])-np.logaddexp.reduce(-fes_ref[phi_ref>0])
        except IOError:
            print('+++ Ref. file not found: '+ref_file+' +++')
    plt.xlim(plot_range)
    plt.ylim(bottom=0, top=max(ymax, max(fes_estimate)))
    plt.xlabel("$\phi$")
    plt.ylabel("FES")
    plt.legend()
    plt.show()
    #references are from OPES
    if DeltaF_T_high is not None:
        print(f'DeltaF at {T_high:4g}K: {DeltaF_T_high:.3f}')
    if DeltaF_T_low is not None:
        print(f'DeltaF at {T_low:4g}K: {DeltaF_T_low:.3f}')
    DeltaF = -np.log(np.count_nonzero(phi>0) / np.count_nonzero(phi<0))
    print('  DeltaF direct: %.3f' % DeltaF)
    if w is not None:
        DeltaF = -np.log(np.sum(w[phi>0]) / np.sum(w[phi<0]))
        print('DeltaF reweight: %.3f' % DeltaF)
    
def plot_ics(name, data, data2=None, chosen=None):
    data = assert_numpy(data)
    if data2 is not None:
        data2 = assert_numpy(data2)
    if chosen is None:
        chosen = np.random.choice(range(len(data[0,:])))
    plt.figure(figsize=(14, 4))
    plt.subplot(1,2,1)
    plt.title(name+' %d of %d'%(chosen+1, len(data[0,:])))
    plt.plot(data[:,chosen], 'o')
    if data2 is not None:
        plt.plot(data2[:,chosen], 'o')
    plt.subplot(1,2,2)
    plt.hist(data[:,chosen], bins=50, histtype='step', label='prior')
    if data2 is not None:
        plt.hist(data2[:,chosen], bins=50, histtype='step', label='mapped')
        plt.legend()
    plt.show()

def plot_all_traj(data, is_swapped=np.zeros(2), n_phi=None):
    if n_phi is None:
        n_phi = md.compute_phi(md.Trajectory(model.positions, model.mdtraj_topology))[1].shape[-1]
    plt.figure(figsize=(16, 4*n_phi))
    for i in range(data.shape[0]):
        for j in range(n_phi):
            plt.subplot(n_phi, data.shape[0], j*data.shape[0]+i+1)
            phi, psi = get_phi_psi(data[i], j)
            end = len(phi)
            plt.scatter(range(end), phi, c=psi, s=10, vmin=-np.pi, vmax=np.pi)
            plt.ylim(-np.pi, np.pi)
            plt.xlim(0, end)
            if len(data) == 2:
                title = 'Target' if i == 0 else 'Prior'
            else:
                title = f'Trajectory {i}'
            plt.title(title)
            plt.ylabel(f'$\phi${j+1}')
            if is_swapped.sum() > 0:
                for sw in np.where(is_swapped)[0]:
                    plt.axvline(sw, c='r', ls=':', alpha=0.1)
    plt.show()

In [ ]:
prior_name = target_name = "AlanineDipeptideImplicit"

In [ ]:
z_matrix = np.array([
    [0, 1, 4, 6],
    [1, 4, 6, 8],
    [2, 1, 4, 0],
    [3, 1, 4, 0],
    [4, 6, 8, 14],
    [5, 4, 6, 8],
    [7, 6, 8, 4], 
    [11, 10, 8, 6],
    [12, 10, 8, 11],
    [13, 10, 8, 11],
    [15, 14, 8, 16],
    [16, 14, 8, 6],
    [17, 16, 14, 15],
    [18, 16, 14, 8],
    [19, 18, 16, 14],
    [20, 18, 16, 19],
    [21, 18, 16, 19]
])
rigid_block = np.array([6, 8, 9, 10, 14])
dim_cartesian = len(rigid_block) * 3 - 6
dim_bonds = len(z_matrix)
dim_angles = dim_bonds
dim_torsions = dim_bonds
coordinate_transform = bg.MixedCoordinateTransformation(
    data=training_data_gen,
    z_matrix=z_matrix,
    fixed_atoms=rigid_block,
    keepdims=dim_cartesian, 
    normalize_angles=True,).to(**ctx)

In [ ]:
bonds, angles, torsions, cartesian, dlogp = coordinate_transform.forward(training_data_gen[:3])
bonds.shape, angles.shape, torsions.shape, cartesian.shape, dlogp.shape

In [ ]:
dim_ics = dim_bonds + dim_angles + dim_torsions + dim_cartesian
mean = torch.zeros(dim_ics).to(**ctx) 
# passing the mean explicitly to create samples on the correct device
prior = bg.NormalDistribution(dim_ics, mean=mean)

In [ ]:
split_into_ics_flow = bg.SplitFlow(dim_bonds, dim_angles, dim_torsions, dim_cartesian)

In [ ]:
# test
_ics = split_into_ics_flow(prior.sample(3))[:-1]
coordinate_transform.forward(*_ics, inverse=True)[0].shape

In [ ]:
shape_info = bg.ShapeDictionary.from_coordinate_transform(
    coordinate_transform)

shape_info

In [ ]:
builder = bg.BoltzmannGeneratorBuilder(
    shape_info, 
    target=target_energy_1, 
    device=ctx["device"], 
    dtype=ctx["dtype"]
)

In [ ]:
from bgflow import TORSIONS, FIXED, BONDS, ANGLES
for i in range(4):
    builder.add_condition(TORSIONS, on=FIXED)
    builder.add_condition(FIXED, on=TORSIONS)
for i in range(2):
    builder.add_condition(BONDS, on=ANGLES)
    builder.add_condition(ANGLES, on=BONDS)
builder.add_map_to_ic_domains()
builder.add_map_to_cartesian(coordinate_transform)
generator = builder.build_generator()

In [ ]:
# TEST
# play forward and backward
samples = generator.sample(2)
energy = generator.energy(samples)
generator.kldiv(10)

In [ ]:
nll_optimizer = torch.optim.Adam(generator.parameters(), lr=1e-3)
nll_trainer = bg.KLTrainer(
    generator, 
    optim=nll_optimizer,
    train_energy=False
)

In [ ]:
def plot_energies(ax, samples, target_energy, test_data):
    sample_energies = target_energy.energy(samples).cpu().detach().numpy()
    md_energies = target_energy.energy(test_data[:len(samples)]).cpu().detach().numpy()
    cut = max(np.percentile(sample_energies, 80), 20)
    
    ax.set_xlabel("Energy   [$k_B T$]")
    # y-axis on the right
    ax2 = plt.twinx(ax)
    ax.get_yaxis().set_visible(False)
    
    ax2.hist(sample_energies, range=(-50, cut), bins=40, density=False, label="BG")
    ax2.hist(md_energies, range=(-50, cut), bins=40, density=False, label="MD")
    ax2.set_ylabel(f"Count   [#Samples / {len(samples)}]")
    ax2.legend()


In [ ]:
if main:
    nll_trainer.train(
        n_iter=5000, 
        data=training_data_gen,
        testdata = test_data,
        batchsize=128,
        n_print=1000, 
        w_energy=0.0
    )

In [ ]:
if main:
    with torch.no_grad():
        n_samples = 10000
        samples = generator.sample(n_samples)

        fig, axes = plt.subplots(1, 3, figsize=(9,3))
        fig.tight_layout()

        # plot_phi_psi(axes[0], test_data, system_1)
        # plot_phi_psi(axes[1], samples, system_1)
        plot_energies(axes[2], samples, target_energy_1, test_data.reshape(-1, dataset_1.dim))
        phi = plot_rama_traj(samples, get_phi=True)
        plot_phi_FES(phi, w=np.ones(len(phi)))
        del samples

In [ ]:

for i,label in enumerate(nll_trainer.reporter._labels):
    
    plt.plot(nll_trainer.reporter._raw[i]) #TODO: better reporter
    plt.title(label)
    plt.show()

### mixed training

In [ ]:
mixed_optimizer = torch.optim.Adam(generator.parameters(), lr=1e-4)
mixed_trainer = bg.KLTrainer(
    generator, 
    optim=mixed_optimizer,
    train_energy=True
)

In [ ]:
# import warnings
# warnings.filterwarnings("ignore", category=UserWarning, message="InputOutsideDomain")
w_energy = 0.1
if main:
    mixed_trainer.train(
        n_iter=500, 
        data=training_data_gen,
        batchsize=1000,
        n_print=100, 
        w_energy=w_energy,
        w_likelihood=1-w_energy,
        clip_forces=100.0
    )

In [ ]:
if main:
    with torch.no_grad():
        n_samples = 10000
        samples = generator.sample(n_samples)

        fig, axes = plt.subplots(1, 3, figsize=(9,3))
        fig.tight_layout()

        # plot_phi_psi(axes[0], test_data, system_1)
        # plot_phi_psi(axes[1], samples, system_1)
        plot_energies(axes[2], samples, target_energy_1, test_data.reshape(-1, dataset_1.dim))
        phi = plot_rama_traj(samples, get_phi=True)
        plot_phi_FES(phi, w=np.ones(len(phi)))
        del samples

In [ ]:
torch.save(generator.state_dict(), 'BGgenerator_T1000_Tip3p.pth')


In [ ]:
generator.load_state_dict(torch.load('BGgenerator_T1000_Tip3p.pth'))
generator.eval()

### Sampling

In [ ]:
batch_size = 30000
total_samples = batch_size * 10

filtered_samples = []
with torch.no_grad():
    for _ in range(total_samples // batch_size):
        samples = generator.sample(batch_size)
        valid_samples = samples
        filtered_samples.append(valid_samples.cpu())
        del samples
        torch.cuda.empty_cache()

final_filtered_samples = torch.cat(filtered_samples, dim=0)

In [ ]:
len(final_filtered_samples)

In [ ]:
if main:
    with torch.no_grad():
        phi = plot_rama_traj(final_filtered_samples, get_phi=True)
        plot_phi_FES(phi, w=np.ones(len(phi)))

In [ ]:
trajectory = md.Trajectory(
    xyz=final_filtered_samples.cpu().detach().numpy().reshape(-1, 22, 3), 
    topology=system_1.mdtraj_topology
    )
trajectory.save_xtc('ala2_BGgen.xtc')

In [ ]:
np.save('ala2_BGgen.npy', final_filtered_samples.cpu().detach().numpy())

# Minimization

In [ ]:
class AlaImplicit(OpenMMSystem):
    def __init__(
            self,
    ):
        super(AlaImplicit, self).__init__()

        forcefield = app.ForceField("amber14-all.xml", 'implicit/gbn2.xml')
        pdb = app.PDBFile("/home/dell/workstations/hsj_bgflow/experiment/ala/Ala2Tip3p1000/ala_no_water.pdb")

        self._system  = forcefield.createSystem(
            pdb.topology,
            nonbondedMethod=app.NoCutoff, 
            constraints=app.HBonds,  
            hydrogenMass=4.0 * amu  
        )

        self._positions = pdb.getPositions(asNumpy=True).value_in_unit(nanometer)
        self._topology = pdb.topology


In [ ]:
from multiprocessing import Pool
from openmm.app import *
from openmm import *
from openmm.unit import *
import numpy as np
import os
from bgmol.systems.base import OpenMMToolsTestSystem

xyz = final_filtered_samples.reshape(-1, 22, 3).cpu().detach().numpy()

def process_sample_cpu(args):
    """
    Optimize a single sample on CPU.
    args: (index, system, topology).
    returns: (index, optimized positions or None, success).
    """
        # if energy > 1e-5:
        #     energy = optimize_until_converged(simulation)
    index, system, topology = args
    try:
        platform = Platform.getPlatformByName('CPU')
        properties = {'Threads': '1'}
        integrator = LangevinMiddleIntegrator(T_1 * kelvin, 1.0 / picosecond, 2.0 * femtosecond)
        simulation = Simulation(topology, system, integrator, platform, properties)
        simulation.context.setPositions(xyz[index])

        energy = simulation.context.getState(getEnergy=True).getPotentialEnergy().value_in_unit(kilojoules_per_mole)
        if energy < 0:
            return index, xyz[index], True
        simulation.step(1000)
        energy = simulation.context.getState(getEnergy=True).getPotentialEnergy().value_in_unit(kilojoules_per_mole)

        optimized_positions = simulation.context.getState(getPositions=True).getPositions().value_in_unit(nanometer)
        
        if energy < 100:
            return index, optimized_positions, True
        else:
            return index, optimized_positions, False

    except Exception as e:
        return index, None, False

if __name__ == "__main__":
    num_cpu_processes = min(48, os.cpu_count())
    print(f"Starting CPU optimization with {num_cpu_processes} processes.")
    system = AlaImplicit()

    cpu_args_list = [(idx, system.system, system.topology) for idx in range(len(xyz))]

    with Pool(processes=num_cpu_processes) as pool:
        cpu_results = pool.map(process_sample_cpu, cpu_args_list)

    optimized_indices = []
    optimized_positions = []

    for index, result, success in cpu_results:
        if success and result is not None:
            optimized_indices.append(index)
            optimized_positions.append(result)
        else:
            pass

    if optimized_positions:
        xyz_new = np.array(optimized_positions)
    else:
        xyz_new = np.empty((0, 22, 3))

    optimized_count = len(optimized_indices)
    failed_count = len(xyz) - optimized_count
    print(f"Optimization completed. Success: {optimized_count}, Failed: {failed_count}")

In [ ]:
np.save('ala2_BGgen_mini.npy', xyz_new)